Python для инженерии данных
## Домашнее задание 4
### Чанышев Тимур ВШЭ-ИД-1-26

### Задача 1
```
Напишите функцию check_car_id(car_id: str) → str.

 Формат номера: 1 буква + 3 цифры + 2 буквы + 2–3 цифры региона.
 Буквы только из набора, допустимого в РФ-номерах: А, В, Е, К, М, Н, О, Р, С, Т, У, Х (кириллица). Учитывайте путаницу с латиницей на входе - нормализуйте или явно опишите правило.

 • Если номер валиден: «Номер А222ВС валиден. Регион: 96.»
 • Если невалиден: «Номер не валиден.»

Пример вывода:

>>> check_car_id('А222ВС96')
 Номер А222ВС валиден. Регион: 96.

 >>> check_car_id('А123ВЕ777')
 Номер А123ВЕ валиден. Регион: 777.

 >>> check_car_id('АБ22ВВ193')
 Номер не валиден.

Пограничные случаи:

>>> check_car_id('A222BC96')   # латиница A/B/C - по правилу нормализации
 Номер А222ВС валиден. Регион: 96.
 # либо «Номер не валиден.» - если нормализацию не делаете; зафиксируйте выбор

 >>> check_car_id('А222ВС9')    # регион из 1 цифры
 Номер не валиден.

 >>> check_car_id('А222ВС9699') # регион из 4 цифр
 Номер не валиден.

 >>> check_car_id('Я222ВС96')   # недопустимая буква Я
 Номер не валиден.

 >>> check_car_id('')
 Номер не валиден.
```

In [81]:
import re

def check_car_id(id):

    # Нормализация
    translate_table = str.maketrans({
        "A": "А",
        "B": "В",
        "E": "Е",
        "K": "К",
        "M": "М",
        "H": "Н",
        "O": "О",
        "P": "Р",
        "C": "С",
        "T": "Т",
        "X": "Х"
    })

    id = id.translate(translate_table)

    allowed_letters = 'АВЕКМНОРСТУХ'

    # Проверка. Регулярное выражение: 1 буква + 3 цифры + 2 буквы + 2 или 3 цифры
    if re.match(rf'^[{allowed_letters}]\d{{3}}[{allowed_letters}]{{2}}\d{{2,3}}$', id):
        return f'Номер {id[:6]} валиден. Регион {id[6:]}'
    else:
        return f'Номер не валиден.'


    

In [82]:
# ТЕСТ
test_ids = [
    'А123ВЕ456',
    'A123BE456', # латиница
    'А13ВЕ456',
    'А222ВС45',
    'А222ВС9699',
    'Я222ВС96',
    ''
]

for test_id in test_ids:
    print(f'{test_id}\t{check_car_id(test_id)}')

А123ВЕ456	Номер А123ВЕ валиден. Регион 456
A123BE456	Номер А123ВЕ валиден. Регион 456
А13ВЕ456	Номер не валиден.
А222ВС45	Номер А222ВС валиден. Регион 45
А222ВС9699	Номер не валиден.
Я222ВС96	Номер не валиден.
	Номер не валиден.


### Задача 2
```
Дан многострочный лог. Реализуйте parse_logs(text: str) → list[dict].

 1. Одной (или составной) регуляркой извлеките строки только в формате:
    YYYY-MM-DD HH:MM:SS LEVEL ...
 2. Из хвоста строки достаньте user, action, ip; если есть card=... - тоже.
 3. Валидация через regex:
    • email (разумный практический паттерн);
    • IPv4;
    • карта: маска XXXX-XXXX-XXXX-XXXX; в результате хранить только ****-****-****-1111 (последние 4 цифры).
 4. Невалидные поля не роняют парсер: в словаре "errors": ["invalid_email", ...].
 5. Бонус: удалите последовательные повторы слов в текстовом поле сообщения через re.

Входной пример:

2024-03-15 14:22:01 INFO user=ivan@mail.ru action=login ip=192.168.0.12
 2024-03-15 14:22:05 WARN user=ivan@mail.ru action=retry ip=192.168.0.12
 2024-03-15 14:23:11 ERROR user=bad..email@ action=pay ip=999.1.1.1 card=4111-1111-1111-1111
 Mar 15 2024 14:24:00 INFO user=olga@company.com action=logout ip=10.0.0.5

Пример вывода:

>>> parse_logs(text)
 [
   {
     'ts': '2024-03-15 14:22:01',
     'level': 'INFO',
     'user': 'ivan@mail.ru',
     'action': 'login',
     'ip': '192.168.0.12',
     'card': None,
     'errors': []
   },
   {
     'ts': '2024-03-15 14:22:05',
     'level': 'WARN',
     'user': 'ivan@mail.ru',
     'action': 'retry',
     'ip': '192.168.0.12',
     'card': None,
     'errors': []
   },
   {
     'ts': '2024-03-15 14:23:11',
     'level': 'ERROR',
     'user': 'bad..email@',
     'action': 'pay',
     'ip': '999.1.1.1',
     'card': '****-****-****-1111',
     'errors': ['invalid_email', 'invalid_ip']
   }
 ]
 # Строка «Mar 15 2024 ...» пропускается - формат даты не YYYY-MM-DD.

Пограничные случаи:

• Пустая строка / только пробелы → [].
 • Строка без user=/action=/ip= → запись с отсутствующими полями (None) и соответствующими errors, либо пропуск - зафиксируйте правило.
 • ip=256.0.0.1 или ip=1.2.3 → errors содержит invalid_ip.
 • card=4111111111111111 без дефисов → не распознаётся как карта (card=None) или invalid_card — зафиксируйте правило.
 • Повторы слов в сообщении: «error error error timeout» → «error timeout» (бонус).
```

In [107]:
import re
from datetime import datetime


def parse_logs(log_file):
    result = []

    log_pattern = re.compile(
        r"^(?P<ts>\d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01]) "
        r"(?:[01]\d|2[0-3]):[0-5]\d:[0-5]\d) "
        r"(?P<level>INFO|WARN|ERROR|DEBUG)\b"
        r"(?P<tail>.*)$"
    )

    email_pattern = re.compile(
        r"(?!.*\.\.)"
        r"[A-Za-z0-9](?:[A-Za-z0-9._%+-]*[A-Za-z0-9])?"
        r"@[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?"
        r"(?:\.[A-Za-z]{2,})+"
    )

    ip_part = r"(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)"
    ip_pattern = re.compile(rf"{ip_part}(?:\.{ip_part}){{3}}")

    card_pattern = re.compile(r"\d{4}(?:-\d{4}){3}")
    action_pattern = re.compile(r"[a-z]{1,10}")

    # Функция поиска доп полей
    def extract(tail, field):
        match = re.search(
            rf"(?:^|\s){re.escape(field)}=([^\s]*)",
            tail
        )
        
        if match:
            return match.group(1)

        return None

    with open(log_file, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            match = log_pattern.fullmatch(line)
            if not match:
                continue

            timestamp = match.group("ts")
            level = match.group("level")
            tail = match.group("tail")

            try:
                datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
            except ValueError:
                continue

            user = extract(tail, "user")
            action = extract(tail, "action")
            ip = extract(tail, "ip")
            card = extract(tail, "card")

            errors = []

            if user is None or not email_pattern.fullmatch(user):
                errors.append("invalid_email")

            if action is None or not action_pattern.fullmatch(action):
                errors.append("invalid_action")

            if ip is None or not ip_pattern.fullmatch(ip):
                errors.append("invalid_ip")

            if card is None:
                masked_card = None
            elif card_pattern.fullmatch(card):
                masked_card = f"****-****-****-{card[-4:]}"
            else:
                masked_card = card
                errors.append("invalid_card")

            item = {
                "ts": timestamp,
                "level": level,
                "user": user,
                "action": action,
                "ip": ip,
                "card": masked_card,
                "errors": errors
            }

            message_match = re.search(r"(?:^|\s)msg=(.*)$", tail)
            if message_match:
                message = message_match.group(1)

                message = re.sub(
                    r"\b(\w+)(?:\s+\1\b)+",
                    r"\1",
                    message,
                    flags=re.IGNORECASE
                )

                item["message"] = message

            result.append(item)

    return result

In [108]:
# ТЕСТ
logs = parse_logs('../src/logs_hw4.txt')

for i, log in enumerate(logs):
    print(log)
    if i == 20:
        break

{'ts': '2024-03-15 10:51:25', 'level': 'WARN', 'user': 'user+tag@gmail.com', 'action': 'login', 'ip': '203.0.113.42', 'card': None, 'errors': []}
{'ts': '2024-03-15 11:46:57', 'level': 'DEBUG', 'user': 'petr.sidorov@yandex.ru', 'action': 'login', 'ip': '8.8.8.8', 'card': None, 'errors': [], 'message': 'retry please'}
{'ts': '2024-03-15 09:36:54', 'level': 'WARN', 'user': 'petr.sidorov@yandex.ru', 'action': 'retry', 'ip': '999.1.1.1', 'card': None, 'errors': ['invalid_ip']}
{'ts': '2024-03-15 09:07:09', 'level': 'ERROR', 'user': 'user@domain.', 'action': 'delete', 'ip': '0.0.0.0', 'card': None, 'errors': ['invalid_email']}
{'ts': '2024-03-15 10:28:11', 'level': 'ERROR', 'user': '@onlydomain.com', 'action': 'pay', 'ip': '300.300.300.300', 'card': '4111-1111-1111-11111', 'errors': ['invalid_email', 'invalid_ip', 'invalid_card']}
{'ts': '2024-03-15 10:04:06', 'level': 'ERROR', 'user': 'user+tag@gmail.com', 'action': 'pay', 'ip': '127.0.0.1', 'card': '4111-1111-1111-1111-1111', 'errors': ['